In [4]:
from google.cloud import bigquery

client = bigquery.Client(project="prod-organize-arizon-4e1c0a83")

query = """
SELECT
*
FROM `prod-organize-arizon-4e1c0a83.viewers_dataset.az_censustract_voters_2024`
"""


df = client.query(query).to_dataframe()

df.head()

,mailaddrcensustract10,population_count,dem_votes,rep_votes,dem_margin,third_votes
0,106002,3280,407,294,1.384354,271
1,061023,13282,2006,2133,0.940459,1908
2,420504,4041,525,679,0.773196,576
3,082211,7112,1175,663,1.772247,903
4,103206,3338,467,906,0.515453,526


In [11]:
import geopandas as gpd

# so didn't know you can do this with raw url until today >:)
url = "https://raw.githubusercontent.com/cmarikos/az-landscape-2024/christina-dev/Analysis%20Files/az_census_tract.geojson"
gdf = gpd.read_file(url)


gdf.crs

<Geographic 2D CRS: EPSG:4269>
Name: NAD83
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: North America - onshore and offshore: Canada - Alberta; British Columbia; Manitoba; New Brunswick; Newfoundland and Labrador; Northwest Territories; Nova Scotia; Nunavut; Ontario; Prince Edward Island; Quebec; Saskatchewan; Yukon. Puerto Rico. United States (USA) - Alabama; Alaska; Arizona; Arkansas; California; Colorado; Connecticut; Delaware; Florida; Georgia; Hawaii; Idaho; Illinois; Indiana; Iowa; Kansas; Kentucky; Louisiana; Maine; Maryland; Massachusetts; Michigan; Minnesota; Mississippi; Missouri; Montana; Nebraska; Nevada; New Hampshire; New Jersey; New Mexico; New York; North Carolina; North Dakota; Ohio; Oklahoma; Oregon; Pennsylvania; Rhode Island; South Carolina; South Dakota; Tennessee; Texas; Utah; Vermont; Virginia; Washington; West Virginia; Wisconsin; Wyoming. US Virgin Islands. British Virgin Islands

In [15]:
# above gdf.crs told me that we're in EPSG:4269
# folium uses EPSG:4326 so we gotta convert

gdf = gdf.to_crs(4326)

In [16]:
# look at all my lovely columns
gdf.head(2)[0:0]

,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry


In [25]:
# now I wanna check that my df mailaddrcensustract10 column matches the GEOID column in gdf
import pandas as pd


df["mailaddrcensustract10"].dtype
gdf["GEOID"].dtype

EXPECTED = 11
left_key  = df["mailaddrcensustract10"].astype("string").str.strip().str.zfill(EXPECTED)
right_key = gdf["GEOID"].astype("string").str.strip().str.zfill(EXPECTED)

any_match = left_key.isin(right_key).any()
any_match

# boo, it dooesn't match "mailaddrcensustract10" is just the tract id and geoid is a full 11 digit geoid
# cripes

False